# RAG с использованием библиотеки LangChain

В этом тюториале мы создадим полноценную RAG-систему с помощью библиотеки _langchain_.

Код и документация библиотек семейства _LangChain_: https://github.com/langchain-ai

Эти библиотеки можно поставить в виртуальное окружение командой `pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-openai`

Импортируем модули которые нам понадобятся впоследствии:

In [ ]:
import logging
from timeit import default_timer as timer

import datasets
import sentence_transformers as sbert

from langchain_classic.chains import RetrievalQA
from langchain_classic.chains import LLMChain
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

Подготовим логгер:

In [ ]:
logging.basicConfig(level=logging.INFO)

## Датасет

Будем работать с простым датасетом RAG Dataset 1200.

Этот датасет удобен тем, что:
- он небольшой, всего 1200 объектов
- содержит развернутые ответы (а не короткие как многие стандартные датасеты для RAG)
- в нем есть эталонные ответы, которые можно использовать для оценки качества

Ссылка: https://huggingface.co/datasets/neural-bridge/rag-dataset-1200

Загрузим датасет:

In [ ]:
start = timer()
data = datasets.load_dataset("neural-bridge/rag-dataset-1200")
logging.info(f"loaded dataset: elapsed = {timer() - start:.3f}")

Проведем EDA:

In [ ]:
print(data)

Посмотрим на один из примеров:

In [ ]:
example_idx = 450

data_train = data['train']
example = data_train[450]
print(example.keys())

Посмотрим на контекст:

In [ ]:
print(example['context'])

Посмотрим на запрос и эталонный ответ:

In [ ]:
print(f"Question: {example['question']}")
print(f"Answer: {example['answer']}")

## Индексация

Распакуем датасет, и сохраним все документы, запросы и эталонные ответы:

In [ ]:
# Unpack dataset and collect docs/queries
docs = []
queries = []
answers = []
for i, example in enumerate(data_train):
    context = example["context"]
    query = example["question"]
    answer = example["answer"]
    docs.append(Document(page_content=context))
    queries.append(query)
    answers.append(answer)
logging.info(f"collected: num_docs = {len(docs)}")

Разобъем текст на чанки с использованием стандартного _RecursiveCharacterTextSplitter_.

In [ ]:
# Make chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)
logging.info(f"splitted into chunks: num_chunks = {len(chunks)}")

Подготовим эмбеддер, который будет использовать GPU, и заэмбеддим наши документы.

Для этого у нас должен быть установлен пакет _sentence_transformers_: ```pip install sentence-transformers```

Скачаем и загрузим модель _MiniLM_:

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Load embedder
embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs = {'device': 'cuda'})
logging.info(f"loaded model: model_name = {model_name}")

Заэмбеддим все чанки, и сохраним эмбеддинги в _faiss_.

Для этого у нас должен быть установлен пакет _faiss_: ```pip install faiss-cpu```

In [ ]:
# Embed all the chunks
start = timer()
vectorstore = FAISS.from_documents(chunks, embeddings)
logging.info(f"embedded chunks: elapsed = {timer() - start:.3f}")

При желании, этот индекс теперь можно было бы сохранить на диск.

## Поиск и ранжирование

Теперь можно попробовать найти в нашем индексе релевантные запросу документы.

Сначала нам придется создать объект-ретривер.

Существуют разные ретриверы:
- dense-ретриверы для поиска по эмбеддингам
- лексические для поиска по ключевым в классическом обратном индексе
- гибридные, которые сочетают достоинства обоих подходов

Мы, для простоты, будем использовать обычный dense.

Найдем в нашем faiss k=10 наиболее близких по косинусному расстоянию документов:

In [ ]:
# Initialize dense retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
logging.info(f"created retriever")

Теперь поднимем с его помощью документы по запросу из нашего примера:

In [ ]:
query = queries[example_idx]
print(f"QUERY: {query}")

# Retrieve documents
retrieved_docs = retriever.invoke(query)
for i, doc in enumerate(retrieved_docs):
    print(f"RETRIEVED: i = {i}")
    print(doc.page_content)

Будем переранжировать поднятые документы с помощью модели кросс-энкодера.

Загрузим стандартный кросс-энкодер на основе _MiniLM_, дообученный на датасете _MS MARCO_:

In [ ]:
ce_model_name = "cross-encoder/ms-marco-MiniLM-L6-v2"

# Load reranker
reranker = sbert.CrossEncoder(ce_model_name)
logging.info(f"loaded reranker model: model_name = {ce_model_name}")

Напишем функцию для переранжирования _top_k_ документов с помощью нашего реранкера:

In [ ]:
def rerank(query, docs, top_k=3):
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    ranked_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked_docs[:top_k]]

Получим топы документов двумя способами, с переранжированием и без него:

In [ ]:
top_k = 3

# Without reranking
top_docs = retrieved_docs[:top_k]
top_docs

In [ ]:
# With reranking
reranked_top_docs = rerank(query, retrieved_docs, top_k=top_k)
reranked_top_docs

## Генерация ответа

Теперь перейдем к самому интересному: используем найденные документы, чтобы сгенерить ответ.

Сначала нам придется подключиться к LLM.

Считаем, что LLM уже поднята с помощью скрипта `llm.sh`.

Давайте в этом убедимся:

In [ ]:
!nvidia-smi

Для взаимодействия с LLM будем использовать стандартный OpenAI API.

Создадим клиента:

In [ ]:
llm_model_name = "Qwen/Qwen3-4B-FP8"

# Create client
llm = ChatOpenAI(openai_api_base=f"http://localhost:8000/v1",
                 openai_api_key="EMPTY",
                 model_name=llm_model_name)
logging.info(f"created LLM client: model_name = {llm_model_name}")

Создадим шаблон для промпта, в который можно будет поставлять наш запрос и контекст (найденные документы):

In [ ]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="Use the context below to answer the question.\nContext: {context}\nQuestion: {question}\nAnswer:")

Создадим объект-чейн, с помощью которого будем делать запросы к LLM:

In [ ]:
chain = LLMChain(llm=llm, prompt=prompt)
logging.info(f"created LLM chain")

Поместим поднятые документы в контекст:

In [ ]:
# From retrieved docs
context = "\n\n".join([doc.page_content for doc in top_docs])

# From retrieved & reranked docs
reranked_context = "\n\n".join([doc.page_content for doc in reranked_top_docs])

Напишем вспомогательную функцию, которая распечатывает ответ LLM:

In [ ]:
def print_rag_answer(query, context, ref_answer, rag_answer):
    # Golden reference answer
    print(f"\n*** Query:\n{query}")
    print(f"\n*** Context:\n{context}")
    print(f"\n*** RAG answer:\n{rag_answer}")
    print(f"\n*** Reference answer:\n{ref_answer}")

Cначала попробуем подать запрос вообще без контекста, и посмотрим что ответит модель.

Тут придется немного подождать пока LLM думает. При желании, можно перейти в консоль и понаблюдать за логами _vllm_.

In [ ]:
query = queries[example_idx]
ref_answer = answers[example_idx]

# Make an LLM call
result = chain.invoke({"context": "", "question": query})
rag_answer = result["text"]
print_rag_answer(query, "", ref_answer, rag_answer)

Видно, что модель отвечает из общих знаний, и, в принципе, отвечает неплохо.

Что будет если мы теперь передадим ей полноценный контекст?
Начнем с варианта без переранжирования:

In [ ]:
# Make an LLM call with retrieved context
result = chain.invoke({"context": context, "question": query})
rag_answer = result["text"]
print_rag_answer(query, context, ref_answer, rag_answer)

Видим, что модель теперь действительно опирается в своем ответе не только на общие знания, но и на контекст!

Аналогично, посмотрим на ответ с контекстом, полученным из переранжированных документов:

In [ ]:
# Make an LLM call with retrieved context
result = chain.invoke({"context": reranked_context, "question": query})
rag_answer = result["text"]
print_rag_answer(query, reranked_context, ref_answer, rag_answer)

Кажется, что ответ модели стал еще более качественным!

Что еще было бы полезно попробовать попробовать/поварьировать/потюнить:
- стратегии чанкования
- векторные индексы (ANN), и их настройки
- ретриверы, напр. гибридный ретривал
- число поднимаемых из индекса документов-кандидатов
- модели-эмбеддеры и модели-реранкеры
- промпт для LLM

Все это вы можете попробовать самостоятельно!